In [ ]:
%load_ext autoreload
%autoreload 2

#global imports
from tensorboard.backend.event_processing import event_accumulator
import os
import scipy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import nn, optim
from torch.nn import functional as F
from torch.utils.tensorboard import SummaryWriter
import matplotlib.pyplot as plt
import copy
from tqdm import tqdm
import csv
from scipy.special import softmax
import random
import logging
logger = logging.getLogger(__name__)
logging.basicConfig(filename='train_log.log', level=logging.INFO)
import itertools
import pandas as pd
import pickle
from sklearn import preprocessing
import pytorch_warmup as warmup

#local imports
from main.Dataset import ImportanceDataset, RealImportanceDataset, RealPredictionDataset, XBoxDatasetSimulation, Axios_ipsosdataset, HouseholdPulse_dataset
from main.GAN import GAN, WGAN_GP
from main.Discriminator import DataDiscriminator, DeepSetCritic
from main.util import set_seed
from DataProcessing import *
from main.experiments import *
from main.analyze_results import *

#variable setups
device = torch.device('cuda:1')

In [ ]:
#REAL DATA - Train RandomGAN on HHP Data
'''
This code trains num_trials number of RandomGAN weights and will save
the results into the folder designated in save_path

'''

#STEP 1 storage path setups
path_to_census_dataset = './data/censusHouseholdPulse_data/cleaned/ipums_cleaned_combined.csv'
path_to_survey_dataset = './data/censusHouseholdPulse_data/cleaned/pulse_week29_cleaned.csv'
save_path = "./saves/"
num_trials = 3
SAVE_DICT = {}
SAVE_DICT['SAVE_DATASET'] = True
SAVE_DICT['SAVE_WEIGHTS'] = True
SAVE_DICT['SAVE_IG'] = False
SAVE_DICT['SAVE_GENERATOR'] = False


#STEP 2 RUN
train(census_dataset_path=path_to_census_dataset,
      survey_dataset_path=path_to_survey_dataset,
      save_path=save_path,
      save_dict = SAVE_DICT,
      num_trials = num_trials,
      train_device=device)

In [ ]:
#REAL DATA - Analyze save directory for results
'''
This cell analyzes the results from training
only need to provide a path to the directory that stores the saved results. 

save_path directory should contain all of the data_X.npt, weight_history_X.npz
as well as the runs directory

'''
num_trials = num_trials #should be the same value as above
save_path = "./saves/"

#Step 1 Path setups
save_final_median(path_to_saves=save_path)

##  Tuning Cell ReadME
### Changing Settings
- If you want to change the data collector:
  - go to ->  product_default_config.yaml > data > dataset_name
  - change to: axios_ipsos, d4p, or household_pulse

- If you want to change the specific survey wave:
  - go to -> product_default_config.yaml > data > weeks
  - change to: ['X'] where X is a valid week for that collector
  - Note that it must be in this exact format with brackets (list) and a string inside ('')
### Tuning Output
This script does not save any local copies of networks or results
or weights. 

It does output two files, saved to disk:
- Results of tuning procedure, as a log text file
  - go to -> hyperparam_tuning/autoTuneDir/generatedFolderName/autotuneLogger.txt
- Saved config with tuned parameters:
  - goto -> configs/tuned_config_(datasetname)_(week_information).yaml

In [ ]:
#Tuning Cell
from hyperparam_tuning.autotuner import notebook_tune_script

notebook_tune_script() #the actual call to the tuning

In [ ]:
#Synthetic Data
path_to_census_dataset = './data/censusHouseholdPulse_data/cleaned/ipums_cleaned_combined.csv'
#the survey data set is not used in synthetic experiments
path_to_survey_dataset = ''
save_path = "./saves_synthetic/"
num_trials = 1
SAVE_DICT = {}
SAVE_DICT['SAVE_DATASET'] = True
SAVE_DICT['SAVE_WEIGHTS'] = False
SAVE_DICT['SAVE_IG'] = False
SAVE_DICT['SAVE_GENERATOR'] = False


#STEP 2 RUN
train_synthetic(census_dataset_path=path_to_census_dataset,
      survey_dataset_path=path_to_survey_dataset,
      save_path=save_path,
      save_dict = SAVE_DICT,
      num_trials = num_trials,
      train_device=device)